In [ ]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
import numpy as np
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

In [ ]:
# -----------------------
# Registers
# -----------------------
q = QuantumRegister(6, "q")  # 2 for psi, 2 for Alice, 2 for Bob
c = ClassicalRegister(6, "c")  # 4 bits for Alice measurement + 2 bits for Bob measurement

qc = QuantumCircuit(q, c)

# Map for clarity
psi = [q[0], q[1]]
A   = [q[2], q[3]]
B   = [q[4], q[5]]

# =====================================================
# 1️⃣ Prepare general 2-qubit state |psi>
# =====================================================
theta = 2 * np.arccos(1/np.sqrt(3))
qc.ry(theta, psi[0])
qc.cx(psi[0], psi[1])
qc.x(psi[1])
qc.h(psi[1])
qc.cx(psi[0], psi[1])
qc.barrier()

# =====================================================
# 2️⃣ Prepare 4D maximally entangled state |Φ+>_AB
# =====================================================
qc.h(A[0])
qc.h(A[1])
qc.cx(A[0], A[1])
qc.cx(A[0], B[0])
qc.cx(A[1], B[1])
qc.barrier()

# =====================================================
# 3️⃣ 4D Bell measurement on psi + Alice
# =====================================================
qc.cx(psi[0], A[0])
qc.cx(psi[1], A[1])
qc.h(psi[0])
qc.h(psi[1])
qc.barrier()

# Measure Alice's qubits (input + entangled)
qc.measure(psi[0], c[0])
qc.measure(psi[1], c[1])
qc.measure(A[0], c[2])
qc.measure(A[1], c[3])
qc.barrier()

# =====================================================
# 4️⃣ Bob's explicit conditional corrections
# =====================================================
# B[0] corrections
with qc.if_test((c[2], 1)):
    qc.x(B[0])
with qc.if_test((c[0], 1)):
    qc.z(B[0])

# B[1] corrections
with qc.if_test((c[3], 1)):
    qc.x(B[1])
with qc.if_test((c[1], 1)):
    qc.z(B[1])

qc.barrier()

# =====================================================
# 5️⃣ Apply inverse of |psi> preparation to Bob (verification)
# =====================================================
qc.cx(B[0], B[1])
qc.h(B[1])
qc.x(B[1])
qc.cx(B[0], B[1])
qc.ry(-theta, B[0])
qc.barrier()

# =====================================================
# 6️⃣ Measure Bob to verify teleportation
# =====================================================
qc.measure(B[0], c[4])
qc.measure(B[1], c[5])

# Draw the circuit
qc.draw("mpl")

In [ ]:
service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
qc_isa = pm.run(qc)
sampler = Sampler(mode=backend)

job = sampler.run([qc_isa])
res = job.result()
counts = res[0].data.c.get_counts()

In [ ]:
total_shots = sum(counts.values())

# Count where first two qubits = 0 (leftmost bits)
count_zero = sum(value for bitstring, value in counts.items() if bitstring[:2] == '00')

# Count where first two qubits ≠ 0
count_nonzero = total_shots - count_zero

# Probability (optional)
prob_zero = count_zero / total_shots
prob_nonzero = count_nonzero / total_shots
print(f"Probability first two qubits = 00: {prob_zero:.4f}")
print(f"Probability first two qubits != 00: {prob_nonzero:.4f}")

# Second histogram data
filtered_counts = {'00': count_zero, 'not 00': count_nonzero}

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

# Full system counts
plot_histogram(counts, ax=ax1, color='#555555')
ax1.set_title("Full System Counts")

# Filtered 2-bar histogram
plot_histogram(filtered_counts, ax=ax2, color='#555555')
ax2.set_title("First two qubits = 00 vs not 00")

plt.tight_layout()
plt.show()